In [1]:
%load_ext autoreload
%autoreload 2

print("Autoreload extension loaded. All modules will be reloaded automatically before executing code.")

Autoreload extension loaded. All modules will be reloaded automatically before executing code.


In [ ]:
import logging
import torch
from torchinfo import summary
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger

from workspace.config import Config
from workspace.dataset import MNISTDataset
from workspace.model import MNISTClassifier

torch.set_float32_matmul_precision("medium")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = WandbLogger(
    save_dir="data/logs",
    name="deepdive",
)

config = Config()
dataset = MNISTDataset(config)

checkpointer = ModelCheckpoint(
    dirpath=config.checkpoint_rootdir,
    filename="mnist-{epoch:02d}",
    save_last=True,
    save_top_k=1,
)

trainer = pl.Trainer(
    accelerator="cuda",
    devices=[0],
    max_epochs=config.epochs,
    callbacks=[checkpointer],
    deterministic=True,
    enable_progress_bar=True,
    enable_model_summary=False,
    logger=logger,
    precision="16-mixed",
)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [3]:
model = MNISTClassifier(
    layer_depth=config.layer_depth,
    kernel_depth=config.kernel_depth,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
)

print("\nLoaded", config, "\n")
summary(model, input_size=(config.batch_size, 1, 28, 28), verbose=1)

trainer.fit(model, datamodule=dataset)

validation_metrics = trainer.validate(model, datamodule=dataset, ckpt_path="last", verbose=False)
print("Validation metrics:", validation_metrics)



Loaded Config {
    "seed": 42,
    "layer_depth": 3,
    "kernel_depth": 64,
    "dataset_rootdir": "data/datasets",
    "dataset_transform": "ToTensor()",
    "num_workers": 4,
    "epochs": 10,
    "batch_size": 2048,
    "learning_rate": 0.0005,
    "weight_decay": 0.0,
    "checkpoint_rootdir": "data/checkpoints",
    "checkpoint_interval": 1,
    "checkpoint_restore": "last",
    "Torch version:": "2.13.0+cu130",
    "CUDA supported:": true
} 

Layer (type:depth-idx)                   Output Shape              Param #
MNISTClassifier                          [2048, 10]                --
├─Sequential: 1-1                        [2048, 10]                --
│    └─Conv2d: 2-1                       [2048, 128, 28, 28]       1,280
│    └─ReLU: 2-2                         [2048, 128, 28, 28]       --
│    └─MaxPool2d: 2-3                    [2048, 128, 14, 14]       --
│    └─Conv2d: 2-4                       [2048, 256, 14, 14]       295,168
│    └─ReLU: 2-5                         

I0000 00:00:1784104021.492269   44195 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784104021.521953   44195 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784104022.172448   44195 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784104022.172684   44195 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
LOCAL_RANK

Output()

/opt/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` 
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/opt/venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (30)
is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you 
want to see logs for the training epoch.

`Trainer.fit` stopped: `max_epochs=10` reached.


Restoring states from the checkpoint path at /workspaces/deepdive/data/checkpoints/last.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /workspaces/deepdive/data/checkpoints/last.ckpt


Output()

Validation metrics: [{'val_loss_epoch': 0.05707451328635216, 'val_acc': 0.9824000000953674}]
